In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib


df = pd.read_excel('كل_التراخيص_مع_شرح_عربي.xlsx') 


df['name_length'] = df['name'].str.len()
df['has_gpl'] = df['licenseId'].str.contains('GPL|AGPL', case=False, na=False).astype(int)
df['has_lgpl'] = df['licenseId'].str.contains('LGPL', case=False, na=False).astype(int)
df['has_mpl'] = df['licenseId'].str.contains('MPL', case=False, na=False).astype(int)
df['has_apache'] = df['licenseId'].str.contains('Apache', case=False, na=False).astype(int)
df['has_bsd'] = df['licenseId'].str.contains('BSD', case=False, na=False).astype(int)
df['has_mit'] = df['licenseId'].str.contains('MIT', case=False, na=False).astype(int)
df['is_osi'] = df['isOsiApproved'].astype(int)
df['is_deprecated'] = df['isDeprecatedLicenseId'].astype(int)


features = ['name_length', 'has_gpl', 'has_lgpl', 'has_mpl', 'has_apache',
            'has_bsd', 'has_mit', 'is_osi', 'is_deprecated']

X = df[features]
y = df['is_safe']   

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=300, random_state=42)
model.fit(X_train, y_train)


pred = model.predict(X_test)
print("دقة المودل البسيط:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))


joblib.dump(model, 'model/license_safety_model_v1.pkl')
print("تم حفظ المودل → license_safety_model_v1.pkl")

دقة المودل البسيط: 0.9020979020979021
              precision    recall  f1-score   support

           0       0.91      0.97      0.94       118
           1       0.82      0.56      0.67        25

    accuracy                           0.90       143
   macro avg       0.87      0.77      0.80       143
weighted avg       0.90      0.90      0.89       143

تم حفظ المودل → license_safety_model_v1.pkl


In [9]:

model = joblib.load('license_safety_model_v1.pkl')

def predict_license_safety(license_id):
    sample = pd.DataFrame([{
        'name_length': len(license_id),
        'has_gpl': 1 if 'GPL' in license_id.upper() else 0,
        'has_lgpl': 1 if 'LGPL' in license_id.upper() else 0,
        'has_mpl': 1 if 'MPL' in license_id.upper() else 0,
        'has_apache': 1 if 'APACHE' in license_id.upper() else 0,
        'has_bsd': 1 if 'BSD' in license_id.upper() else 0,
        'has_mit': 1 if 'MIT' in license_id.upper() else 0,
        'is_osi': 1,    
        'is_deprecated': 0
    }])
    pred = model.predict(sample)[0]
    prob = model.predict_proba(sample)[0]
    return "آمن" if pred == 1 else "خطر", max(prob)

print(predict_license_safety("MIT"))        
print(predict_license_safety("GPL-3.0"))    
print(predict_license_safety("Apache-2.0")) 



('آمن', np.float64(0.9506111111111111))
('خطر', np.float64(0.913755291005291))
('آمن', np.float64(0.8051666666666668))
